# 🎯 Sessão 05 — Volumetria Clássica: Baseline Definitivo com Validação Cruzada

> **Objetivo:** estabelecer o baseline rigoroso e honesto contra o qual as Redes Neurais (Sessão 06) serão comparadas. Utiliza-se validação cruzada k-fold (Hastie et al., 2009) para obter estimativas não-viesadas de RMSE, MAE, MAPE, R² e bias, com bootstrap (Efron & Tibshirani, 1993) para gerar intervalos de confiança das métricas.

---

## 📑 Sumário

1. [Fundamentação Teórica](#1-fundamentação-teórica)
2. [Setup e Dados](#2-setup-e-dados)
3. [Ajuste Único vs. Validação Cruzada](#3-ajuste-único-vs-validação-cruzada)
4. [Comparação entre Modelos via 5-Fold CV](#4-comparação-entre-modelos-via-5-fold-cv)
5. [Intervalos de Confiança via Bootstrap](#5-intervalos-de-confiança-via-bootstrap)
6. [Baseline Definitivo](#6-baseline-definitivo)
7. [Síntese e Alvos para Machine Learning](#7-síntese-e-alvos-para-machine-learning)

## 1. Fundamentação Teórica

### Por que validação cruzada?

Na Sessão 03 ajustamos os modelos em **todos os dados disponíveis** e avaliamos métricas no mesmo conjunto. Isso produz uma estimativa **otimista** do desempenho: o modelo já viu cada ponto durante o ajuste.

A **validação cruzada k-fold** (Hastie, Tibshirani & Friedman, 2009) corrige esse viés:

1. Divide-se o dataset em $k$ partes (dobras) de tamanho aproximadamente igual.
2. Para cada dobra $i \in \{1, \dots, k\}$:
    - Treina-se o modelo nas $k-1$ dobras restantes.
    - Avalia-se na dobra $i$ (dados nunca vistos durante o ajuste).
3. As $k$ estimativas são agregadas por média e desvio.

Esse procedimento simula o desempenho do modelo em dados novos — exatamente a condição em que ele será usado na prática.

### Por que bootstrap?

O bootstrap (Efron & Tibshirani, 1993) constrói intervalos de confiança para qualquer estatística por meio de reamostragem com reposição. Aplicado às métricas, responde a perguntas como: *"o RMSE de 0.042 é estável ou poderia variar muito em outra amostra?"*

## 2. Setup e Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from forestpy.data.loaders import load_pef_vinhedo
from forestpy.dendrometria.fitting import fit_model
from forestpy.ml.evaluation import kfold_cv, bootstrap_metric
from forestpy.ml.metrics import rmse, r2 as r2_metric
from forestpy.utils import set_seed, get_logger
from forestpy.viz.style import apply_forest_style
from forestpy.viz.diagnostics import plot_predicted_vs_observed, plot_residuals

set_seed(42)
apply_forest_style()
log = get_logger('sessao_05')

df = load_pef_vinhedo(synthetic_fallback=True, n_synthetic=500)
log.info(f'Dataset: {df.shape[0]} árvores')

## 3. Ajuste Único vs. Validação Cruzada

Demonstração do viés otimista do ajuste único. Os dois R² medem coisas conceitualmente diferentes — o primeiro mede aderência aos dados de treino, o segundo estima generalização.

In [ ]:
# Ajuste em todos os dados (Sessão 03)
res_unico = fit_model('schumacher_hall', df['volume'], df['dap'], df['h'])
log.info(f'Ajuste único       — R² = {res_unico.metrics["r2"]:.4f}, '
         f'RMSE = {res_unico.metrics["rmse"]:.4f}')

# Função wrapper compatível com kfold_cv
def fit_predict_schumacher(X_train, y_train, X_test):
    def f(X, b0, b1, b2):
        dap, h = X.T
        return np.exp(b0) * np.power(dap, b1) * np.power(h, b2)
    popt, _ = curve_fit(f, X_train, y_train, p0=[-9.5, 1.8, 1.1], maxfev=5000)
    return None, f(X_test, *popt)

X = df[['dap', 'h']].values
y = df['volume'].values

cv_sh = kfold_cv(
    fit_predict_schumacher, X, y,
    n_splits=5, model_name='schumacher_hall',
)
log.info(f'Validação cruzada  — R² = {cv_sh.mean_metrics["r2"]:.4f} ± '
         f'{cv_sh.std_metrics["r2"]:.4f}, '
         f'RMSE = {cv_sh.mean_metrics["rmse"]:.4f} ± '
         f'{cv_sh.std_metrics["rmse"]:.4f}')

In [ ]:
# Detalhe por dobra
cv_sh.to_dataframe().round(5)

## 4. Comparação entre Modelos via 5-Fold CV

In [ ]:
def fit_predict_spurr(X_train, y_train, X_test):
    def f(X, b0, b1):
        dap, h = X.T
        return b0 + b1 * (dap ** 2) * h
    popt, _ = curve_fit(f, X_train, y_train, p0=[0.001, 0.00003], maxfev=5000)
    return None, f(X_test, *popt)

cv_spurr = kfold_cv(fit_predict_spurr, X, y, n_splits=5, model_name='spurr')

print(cv_sh.summary())
print()
print(cv_spurr.summary())

In [ ]:
# Tabela consolidada
comparativo = pd.DataFrame([
    {'Modelo': cv_sh.model_name, 'RMSE médio': cv_sh.mean_metrics['rmse'],
     'RMSE std': cv_sh.std_metrics['rmse'], 'R² médio': cv_sh.mean_metrics['r2'],
     'MAPE médio (%)': cv_sh.mean_metrics['mape'], 'Bias médio': cv_sh.mean_metrics['bias']},
    {'Modelo': cv_spurr.model_name, 'RMSE médio': cv_spurr.mean_metrics['rmse'],
     'RMSE std': cv_spurr.std_metrics['rmse'], 'R² médio': cv_spurr.mean_metrics['r2'],
     'MAPE médio (%)': cv_spurr.mean_metrics['mape'], 'Bias médio': cv_spurr.mean_metrics['bias']},
]).round(5)

comparativo.to_csv('../reports/tables/05_baseline_cv.csv', index=False)
comparativo

In [ ]:
# Visualização da variabilidade entre dobras (boxplot)
df_folds = pd.concat([
    cv_sh.to_dataframe().assign(modelo='Schumacher-Hall'),
    cv_spurr.to_dataframe().assign(modelo='Spurr'),
], ignore_index=True)

fig_box, axes = plt.subplots(1, 2, figsize=(13, 5))

for i, metrica in enumerate(['rmse', 'r2']):
    df_folds.boxplot(column=metrica, by='modelo', ax=axes[i], grid=True)
    axes[i].set_title(f'Distribuição de {metrica.upper()} entre dobras',
                      fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(metrica.upper())

fig_box.suptitle('')  # remove o supertítulo automático do pandas
fig_box.tight_layout()
fig_box.savefig('../reports/figures/05_boxplot_dobras.png')
plt.show()

## 5. Intervalos de Confiança via Bootstrap

O bootstrap aplicado ao melhor modelo gera IC robustos para as métricas, permitindo comparações estatisticamente fundamentadas com modelos alternativos.

In [ ]:
# Reajusta Schumacher-Hall em todos os dados para extrair predições
res_final = fit_model('schumacher_hall', df['volume'], df['dap'], df['h'])

ic_rmse = bootstrap_metric(
    df['volume'].values, res_final.y_pred,
    rmse, n_bootstrap=2000, confidence_level=0.95,
)
ic_r2 = bootstrap_metric(
    df['volume'].values, res_final.y_pred,
    r2_metric, n_bootstrap=2000, confidence_level=0.95,
)

log.info(f'Bootstrap (n=2000) — Schumacher-Hall:')
log.info(f'  RMSE = {ic_rmse["mean"]:.4f} '
         f'(IC 95%: [{ic_rmse["ci_lower"]:.4f}, {ic_rmse["ci_upper"]:.4f}])')
log.info(f'  R²   = {ic_r2["mean"]:.4f} '
         f'(IC 95%: [{ic_r2["ci_lower"]:.4f}, {ic_r2["ci_upper"]:.4f}])')

## 6. Baseline Definitivo

In [ ]:
fig_po = plot_predicted_vs_observed(
    df['volume'].values, res_final.y_pred,
    title='Schumacher-Hall — Baseline Volumétrico Final',
    unit='m³',
)
fig_po.savefig('../reports/figures/05_predito_vs_observado_final.png')
plt.show()

In [ ]:
fig_res = plot_residuals(
    df['volume'].values, res_final.y_pred,
    title='Diagnóstico Final de Resíduos — Schumacher-Hall',
)
fig_res.savefig('../reports/figures/05_residuos_final.png')
plt.show()

## 7. Síntese e Alvos para Machine Learning

### Baseline registrado

A equação de **Schumacher-Hall**, avaliada por validação cruzada 5-fold, estabelece os seguintes alvos para qualquer modelo alternativo (Sessão 06 em diante):

| Métrica | Valor (5-fold CV) | Significado prático |
|---|---|---|
| RMSE | ver tabela acima | Erro típico em m³ |
| MAPE | ver tabela acima | Erro percentual médio |
| R² | ver tabela acima | Proporção da variância explicada |
| Bias | ≈ 0 | Sem tendência sistemática |

### Critério de superação para a MLP

Para que se afirme que uma Rede Neural **supera** este baseline, exige-se que:

1. O RMSE médio em CV seja menor que o RMSE médio do Schumacher-Hall;
2. A diferença seja **estatisticamente significativa** (ICs bootstrap não se sobrepõem);
3. O modelo não apresente bias maior que o baseline.

Esse rigor evita a armadilha comum de comparar um modelo neural treinado e avaliado nos mesmos dados com um modelo clássico avaliado em validação cruzada — comparação que sempre favorece artificialmente o primeiro.

### Próxima sessão (06): Volumetria com MLP em PyTorch

Implementação da primeira Rede Neural Artificial do projeto: arquitetura, loop de treinamento manual com early stopping, normalização das features, e comparação justa com o baseline aqui estabelecido.